In [9]:
################### IMPORT MODULES #######################
import cv2 as cv
import numpy as np
import os

In [10]:
def hough_transform(edge):
    lines = cv.HoughLinesP(edge, 1, np.pi / 180, 50, minLineLength=50, maxLineGap=150)
    # print(lines)
    # print(len(lines))
    
    line_image = np.zeros_like(resize_)
    road_mask = np.zeros_like(resize_)

    left_lines = []
    right_lines = []

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            slope = (y2 - y1) / (x2 - x1) if x2 - x1 != 0 else np.inf 
            
            if slope < 0: 
                left_lines.append((x1, y1))
                left_lines.append((x2, y2))
            else:  
                right_lines.append((x1, y1))
                right_lines.append((x2, y2))
            
            cv.line(line_image, (x1, y1), (x2, y2), (0, 255, 255), 2)

    if len(left_lines) > 1 and len(right_lines) > 1:
        left_lines = sorted(left_lines, key=lambda point: point[1]) 
        right_lines = sorted(right_lines, key=lambda point: point[1], reverse=True)

        polygon = np.array(left_lines + right_lines, np.int32)
        polygon = polygon.reshape((-1, 1, 2))

        cv.fillPoly(road_mask, [polygon], (0, 255, 0))

    return cv.addWeighted(resize_, 1, road_mask, 0.5, 0)

In [11]:
# cwd = os.getcwd()
# path = os.path.join(cwd, 'Data', '4608279-uhd_3840_2160_24fps.mp4')
# capture = cv.VideoCapture(path)

# if not capture.isOpened():
#     print("error")
#     exit()
# while True:
#     isTrue,frame = capture.read()
#     resize_ = cv.resize(frame,(1200,800),interpolation = cv.INTER_LINEAR)
#     grayScale = cv.cvtColor(resize_, cv.COLOR_BGR2GRAY)
#     GaussianBlur = cv.GaussianBlur(grayScale, (7,7), 0)
#     edge = cv.Canny(GaussianBlur,75,150)
#     cv.imshow("sfsdg",edge)
#     cv.imshow("GaussianBlur",GaussianBlur)    
#     if cv.waitKey(20) & 0xFF ==ord("q"):
#         break
# capture.release()
# cv.destroyAllWindows()

In [12]:
cwd = os.getcwd()
path = os.path.join(cwd, 'Data', '4695859-uhd_3840_2160_30fps.mp4')
capture = cv.VideoCapture(path)

white_range = np.array([[70, 10, 50], [90, 60, 255]])
# white_range = np.array([[0, 0, 200], [180, 55, 255]])
yellow_range = np.array([[0, 0, 0], [0, 0, 255]])
# yellow_range = np.array([[15, 150, 20], [35, 255, 255]])

if not capture.isOpened():
    print("Error: Unable to open video.")
    exit()

# Corrected ROI coordinates to fit within (1200, 800)
roi_x1, roi_y1, roi_x2, roi_y2 = 1, 411, 1199, 800

while True:
    isTrue, frame = capture.read()
    if not isTrue:
        break
    
    resize_ = cv.resize(frame, (1200, 800), interpolation=cv.INTER_LINEAR)

    hsv_image = cv.cvtColor(resize_, cv.COLOR_BGR2HSV)
    
    white_mask = cv.inRange(hsv_image, white_range[0], white_range[1])
    yellow_mask = cv.inRange(hsv_image, yellow_range[0], yellow_range[1])
    combined_mask = cv.bitwise_or(white_mask, yellow_mask)
    
    mask_with_roi = np.zeros_like(combined_mask)
    mask_with_roi[roi_y1:roi_y2, roi_x1:roi_x2] = combined_mask[roi_y1:roi_y2, roi_x1:roi_x2]
    
    result = cv.bitwise_and(resize_, resize_, mask=mask_with_roi)
    
    GaussianBlur = cv.GaussianBlur(result, (5, 5), 0)
    edge = cv.Canny(GaussianBlur, 75, 150)
    
    road_overlay = hough_transform(edge)

    # Display results
    cv.imshow("Detected lane", road_overlay)
    # cv.imshow("Original Resized Frame", resize_)
    # cv.imshow("Masked Region Result", result)
    # cv.imshow("Detected Edges", edge)
    # cv.imshow("Detected Lane Lines", line_image)

    if cv.waitKey(20) & 0xFF == ord("q"):
        break

capture.release()
cv.destroyAllWindows()
